# 03B — CIC-IDS2017: Preprocessing, Expanded Features & Group-Aware Splitting

**NIDS-SE-AT-CNN | Kaggle | New experimental protocol**

Mục tiêu: chuyển từ bộ **19 feature pilot** sang bộ đặc trưng flow rộng hơn; kiểm tra rõ **Web Attack – Brute Force, XSS, SQL Injection**; tạo **train / validation / test** có `_source_file`, `_source_row` để truy vết và nhóm các dòng có vector feature hoàn toàn giống nhau vào cùng một split.

**Quy tắc nghiên cứu:**
- Đây là **protocol mới**; không so sánh trực tiếp điểm với Notebook 04/05 V1–V5 trên split cũ.
- Chỉ học imputation/feature-selection từ `train`; chỉ chọn mô hình trên `validation`. Test chỉ dùng sau khi chốt mô hình; notebook 03B **không đánh giá trên test**.
- Grouping theo vector feature giảm *exact-duplicate leakage* nhưng **không loại trừ event, day hoặc host leakage**: CSV không có IP/timestamp. Nên cần tập dữ liệu độc lập cho kiểm chứng cuối.
- Mẫu theo lớp là **pilot cân bằng có chủ đích**, không thể suy diễn FPR/Precision ngoài thực tế từ tỷ lệ lớp ở đây.
- Một số lớp rất hiếm (SQL Injection, Heartbleed, Infiltration), các chỉ số có độ bất định cao.

## 0. Cấu hình Kaggle & xác định 8 file CSV

In [82]:
from pathlib import Path
import json, hashlib, time, warnings, sys
from collections import Counter
import numpy as np
import pandas as pd

SEED = 42
CHUNK_SIZE = 50_000
MAX_PER_ATTACK = 3_500   # mỗi nhãn tấn công, giữ toàn bộ nếu số mẫu ít hơn
MAX_BENIGN = 7_000
MIN_CLASS_SUPPORT = 2
OUTPUT = Path('/kaggle/working/nids_outputs/03B_CIC_Feature_Engineering') if Path('/kaggle/working').exists() else Path('reports/notebook_outputs/03B_CIC_Feature_Engineering')
OUTPUT.mkdir(parents=True, exist_ok=True)

DATA_ROOT = None  # có thể chỉ định Path('/kaggle/input/datasets/chienvoo/nids-se-at-cnn-data/CIC-IDS2017')
SEARCH_ROOTS = [Path('/kaggle/input'), Path('/mnt/data')]
if DATA_ROOT is not None:
    search_roots = [Path(DATA_ROOT)]
else:
    search_roots = [p for p in SEARCH_ROOTS if p.exists()]

FILES = sorted({p.resolve() for root in search_roots for p in root.rglob('*.csv')
                if 'workinghours' in p.name.lower() and 'iscx' in p.name.lower()})
if len(FILES) != 8:
    raise FileNotFoundError(f'Cần đúng 8 file CIC-IDS2017 CSV, thấy {len(FILES)}: {[str(x) for x in FILES]}. Thêm dataset nids-se-at-cnn-data qua Add Input.')
print('CSV sources:', len(FILES), '\n'+'\n'.join(str(p) for p in FILES))
print('Output:', OUTPUT)

CSV sources: 8 
/kaggle/input/datasets/chienvoo/nids-se-at-cnn-data/CIC-IDS2017/Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv
/kaggle/input/datasets/chienvoo/nids-se-at-cnn-data/CIC-IDS2017/Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv
/kaggle/input/datasets/chienvoo/nids-se-at-cnn-data/CIC-IDS2017/Friday-WorkingHours-Morning.pcap_ISCX.csv
/kaggle/input/datasets/chienvoo/nids-se-at-cnn-data/CIC-IDS2017/Monday-WorkingHours.pcap_ISCX.csv
/kaggle/input/datasets/chienvoo/nids-se-at-cnn-data/CIC-IDS2017/Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv
/kaggle/input/datasets/chienvoo/nids-se-at-cnn-data/CIC-IDS2017/Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv
/kaggle/input/datasets/chienvoo/nids-se-at-cnn-data/CIC-IDS2017/Tuesday-WorkingHours.pcap_ISCX.csv
/kaggle/input/datasets/chienvoo/nids-se-at-cnn-data/CIC-IDS2017/Wednesday-workingHours.pcap_ISCX.csv
Output: /kaggle/working/nids_outputs/03B_CIC_Feature_Engineering


## 1. Schema audit — chỉ giữ feature số được ghi nhận, không dùng metadata làm đầu vào

Không dùng source file, row index, nhãn hoặc trường định danh làm input. `Destination Port` được giữ thành một **biến thể feature có/không** vì có thể mang shortcut phụ thuộc môi trường capture. Giữ nhãn gốc, không tự động hợp nhất Web Attack.

In [83]:
def canonical_label(v):
    x = str(v).strip().replace('\ufffd', '-').replace('  ', ' ')
    return ' '.join(x.split())

headers = []
for p in FILES:
    h = pd.read_csv(p, nrows=0, encoding='latin1').columns.tolist()
    h = [str(x).strip() for x in h]
    if len(h) != len(set(h)):
        raise ValueError(f'Duplicate column headers in {p.name}: {h}')
    headers.append(h)
if any(x != headers[0] for x in headers):
    raise ValueError('CSV headers differ between source files: do not silently align columns')

all_columns = headers[0]
assert 'Label' in all_columns
banned = {'Label','Flow ID','Source IP','Destination IP','Timestamp','Source Port','Unnamed: 0'}
raw_features = [f for f in all_columns if f not in banned and not f.startswith('Unnamed')]
# CIC-IDS2017: Fwd Header Length.1 is a known duplicate, remove as an explicit rule.
if 'Fwd Header Length.1' in raw_features:
    raw_features.remove('Fwd Header Length.1')

print(f'{len(all_columns)} raw columns, {len(raw_features)} candidate numerical features')
print('Selected feature candidates:', raw_features)

79 raw columns, 77 candidate numerical features
Selected feature candidates: ['Destination Port', 'Flow Duration', 'Total Fwd Packets', 'Total Backward Packets', 'Total Length of Fwd Packets', 'Total Length of Bwd Packets', 'Fwd Packet Length Max', 'Fwd Packet Length Min', 'Fwd Packet Length Mean', 'Fwd Packet Length Std', 'Bwd Packet Length Max', 'Bwd Packet Length Min', 'Bwd Packet Length Mean', 'Bwd Packet Length Std', 'Flow Bytes/s', 'Flow Packets/s', 'Flow IAT Mean', 'Flow IAT Std', 'Flow IAT Max', 'Flow IAT Min', 'Fwd IAT Total', 'Fwd IAT Mean', 'Fwd IAT Std', 'Fwd IAT Max', 'Fwd IAT Min', 'Bwd IAT Total', 'Bwd IAT Mean', 'Bwd IAT Std', 'Bwd IAT Max', 'Bwd IAT Min', 'Fwd PSH Flags', 'Bwd PSH Flags', 'Fwd URG Flags', 'Bwd URG Flags', 'Fwd Header Length', 'Bwd Header Length', 'Fwd Packets/s', 'Bwd Packets/s', 'Min Packet Length', 'Max Packet Length', 'Packet Length Mean', 'Packet Length Std', 'Packet Length Variance', 'FIN Flag Count', 'SYN Flag Count', 'RST Flag Count', 'PSH Flag 

## 2. Lấy mẫu ngẫu nhiên có giới hạn RAM, ghi định danh mẫu gốc

**Priority sampling per class:** đọc từng chunk, gán khóa ngẫu nhiên cho từng dòng, giữ `K` khóa thấp nhất cho mỗi nhãn. Điều này không chỉ lấy những dòng đầu tiên trong file và không tự ý oversample các lớp hiếm.

`_source_row` là vị trí dòng dữ liệu **0-based sau header** trong file gốc. Các cột `_source_*` chỉ dùng audit, tuyệt đối không dùng huấn luyện.

In [84]:
rng = np.random.default_rng(SEED)
kept = {}
source_counts = Counter()
read_log = []
t0 = time.time()
for path in FILES:
    offset = 0
    for chunk_idx, chunk in enumerate(pd.read_csv(path, chunksize=CHUNK_SIZE, encoding='latin1', low_memory=False)):
        chunk.columns = [str(x).strip() for x in chunk.columns]
        if not set(raw_features + ['Label']).issubset(chunk.columns):
            raise ValueError(f'Missing features in {path.name}')
        chunk['Label'] = chunk['Label'].map(canonical_label)
        chunk['_source_file'] = path.name
        chunk['_source_row'] = np.arange(offset, offset + len(chunk), dtype=np.int64)
        chunk['_priority'] = rng.random(len(chunk))
        offset += len(chunk)
        for label, part in chunk.groupby('Label', sort=False):
            source_counts[label] += len(part)
            max_keep = MAX_BENIGN if label == 'BENIGN' else MAX_PER_ATTACK
            part = part.nsmallest(max_keep, '_priority')
            if label in kept:
                kept[label] = pd.concat([kept[label],part],ignore_index=True).nsmallest(max_keep,'_priority')
            else:
                kept[label] = part.copy()
    read_log.append({'file':path.name,'rows':offset})
    print(path.name, f'{offset:,} rows')

sample = pd.concat(list(kept.values()),ignore_index=True).drop(columns=['_priority'])
sample['Label'] = sample['Label'].astype(str)
assert sample[['_source_file','_source_row']].duplicated().sum() == 0
print(f'Sampled {len(sample):,} / {sum(source_counts.values()):,} rows in {(time.time()-t0):.1f}s; {sample.Label.nunique()} classes')
print(sample.Label.value_counts().to_string())

Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv 225,745 rows
Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv 286,467 rows
Friday-WorkingHours-Morning.pcap_ISCX.csv 191,033 rows
Monday-WorkingHours.pcap_ISCX.csv 529,918 rows
Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv 288,602 rows
Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv 170,366 rows
Tuesday-WorkingHours.pcap_ISCX.csv 445,909 rows
Wednesday-workingHours.pcap_ISCX.csv 692,703 rows
Sampled 39,193 / 2,830,743 rows in 26.0s; 15 classes
Label
BENIGN                          7000
DDoS                            3500
PortScan                        3500
DoS Slowhttptest                3500
DoS slowloris                   3500
SSH-Patator                     3500
FTP-Patator                     3500
DoS GoldenEye                   3500
DoS Hulk                        3500
Bot                             1966
Web Attack ï¿½ Brute Force      1507
Web Attack ï¿½ XSS               652
Infiltration                

## 3. Làm sạch kiểu dữ liệu, ghi nhận missing/Inf và nhóm exact duplicates

Chỉ chuyển kiểu ở feature; không dùng nhãn để tạo group key. Tất cả bản ghi có vector đặc trưng giống hệt (sau numeric conversion) sẽ mang cùng `_duplicate_group` và không được đi qua nhiều split. Các mẫu feature giống nhau nhưng có nhãn khác nhau được báo cáo như **label conflicts**, không xóa âm thầm.

In [85]:
X_raw = sample[raw_features].apply(pd.to_numeric,errors='coerce')
X_raw = X_raw.replace([np.inf,-np.inf], np.nan).astype('float32')
quality = pd.DataFrame({
    'feature':raw_features,
    'missing_or_nonfinite':X_raw.isna().sum().to_numpy(),
    'missing_ratio':X_raw.isna().mean().to_numpy(),
    'unique_nonmissing':X_raw.nunique(dropna=True).to_numpy(),
}).sort_values(['missing_ratio','feature'], ascending=[False,True])
quality.to_csv(OUTPUT/'feature_quality_audit.csv',index=False)
print('Non-finite cells:',int(X_raw.isna().sum().sum()))
display(quality.head(15))

# Group hash uses only feature values. Hash collisions are exceptionally unlikely;
# for high-assurance deduplication, verify grouped rows across all columns.
group_hash = pd.util.hash_pandas_object(X_raw, index=False).to_numpy(dtype='uint64')
sample['_duplicate_group'] = group_hash
sample['_binary_label'] = (sample.Label!='BENIGN').astype('int8')
conflicts = (sample.groupby('_duplicate_group')['Label'].nunique()>1)
print('Exact-feature duplicate rows:',int(sample.duplicated('_duplicate_group').sum()))
print('Groups with conflicting labels:',int(conflicts.sum()))
if conflicts.any():
    display(sample.loc[sample._duplicate_group.isin(conflicts[conflicts].index),
                       ['Label','_source_file','_source_row','_duplicate_group']].head(20))

Non-finite cells: 90


,feature,missing_or_nonfinite,missing_ratio,unique_nonmissing
14,Flow Bytes/s,45,0.001148,22691
15,Flow Packets/s,45,0.001148,27161
47,ACK Flag Count,0,0.000000,2
71,Active Max,0,0.000000,6710
69,Active Mean,0,0.000000,6787
72,Active Min,0,0.000000,5734
70,Active Std,0,0.000000,1873
52,Average Packet Size,0,0.000000,4967
54,Avg Bwd Segment Size,0,0.000000,1751
53,Avg Fwd Segment Size,0,0.000000,3090


Exact-feature duplicate rows: 3586
Groups with conflicting labels: 0


## 4. Group-aware train/validation/test 60/20/20, phân lớp ưu tiên

Áp dụng `StratifiedGroupKFold` trên nhãn **15 lớp**, với nhóm feature giống nhau. 3 fold cho train, 1 validation, 1 test. Split có thể không đều vì group constraints và lớp cực hiếm; in đầy đủ support để tránh hiểu nhầm.

**Quan trọng:** Bước chia tập sử dụng nhãn để stratify, nhưng **không sử dụng test để lựa chọn feature, model hay hyperparameter**. Đây vẫn là protocol *pilot* mới, chưa chống event leakage.

In [86]:
from sklearn.model_selection import StratifiedGroupKFold

splitter = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)
# SGKF output is train/holdout for each fold; each row gets exactly one fold ID.
fold = np.full(len(sample), -1, dtype=np.int8)
for k, (_, hold) in enumerate(splitter.split(X_raw, sample['Label'],groups=sample['_duplicate_group'])):
    if np.any(fold[hold]!=-1): raise AssertionError('Fold assignment overlap')
    fold[hold] = k
assert (fold>=0).all()
split = np.where(fold==0,'test',np.where(fold==1,'val','train'))
sample['_split'] = split
assert sample.groupby('_duplicate_group')['_split'].nunique().max()==1
assert sample.groupby(['_source_file','_source_row']).size().max()==1
support = pd.crosstab(sample['Label'],sample['_split']).reindex(columns=['train','val','test'],fill_value=0)
support['total'] = support.sum(axis=1)
support.to_csv(OUTPUT/'class_split_support.csv')
print('Split counts:',sample['_split'].value_counts().to_dict())
display(support)
if (support['train']==0).any():
    raise RuntimeError('A class has zero training support. Reduce grouping constraints, adjust sampling or use an explicit rare-class policy.')
if (support[['val','test']]==0).any().any():
    warnings.warn('One or more rare classes have no validation/test samples. Report unsupported class metrics explicitly; do not fabricate them.')

Split counts: {'train': 23922, 'val': 7862, 'test': 7409}


_split,train,val,test,total
Label,,,,
BENIGN,4279,1368,1353,7000
Bot,1190,389,387,1966
DDoS,2107,712,681,3500
DoS GoldenEye,2096,723,681,3500
DoS Hulk,2136,779,585,3500
DoS Slowhttptest,2186,687,627,3500
DoS slowloris,2077,724,699,3500
FTP-Patator,2259,683,558,3500
Heartbleed,8,1,2,11


## 5. Feature filtering (TRAIN ONLY)

Loại feature hoàn toàn rỗng, gần như rỗng (trên 99.5% NA ở train), constant và duplicate *trùng cột theo nội dung* trên train. Giữ hai bộ feature **with_port** và **without_port** để kiểm tra tác dụng của `Destination Port`. Các phép median imputation và scaling được fit **sau khi chọn split**, ở các notebook huấn luyện hoặc ở phép thử validation bên dưới.

In [87]:
train_mask = sample['_split'].eq('train').to_numpy()
train_X = X_raw.loc[train_mask]
max_missing = 0.995
eligible = [f for f in raw_features if train_X[f].isna().mean() < max_missing and train_X[f].nunique(dropna=True)>1]
# Deduplicate feature columns only on train, use fingerprints then verify equality.
selected=[]
for f in eligible:
    if any(train_X[f].equals(train_X[g]) for g in selected):
        print('Dropped duplicate feature column:',f)
    else: selected.append(f)
features_with_port = selected
features_without_port = [f for f in selected if f!='Destination Port']
assert len(features_without_port)>0
print('Selected with port:',len(features_with_port),'without port:',len(features_without_port))
(OUTPUT/'feature_sets.json').write_text(json.dumps({'with_port':features_with_port,'without_port':features_without_port},indent=2),encoding='utf-8')

Dropped duplicate feature column: SYN Flag Count
Dropped duplicate feature column: ECE Flag Count
Dropped duplicate feature column: Avg Fwd Segment Size
Dropped duplicate feature column: Avg Bwd Segment Size
Dropped duplicate feature column: Subflow Fwd Packets
Dropped duplicate feature column: Subflow Fwd Bytes
Dropped duplicate feature column: Subflow Bwd Packets
Selected with port: 60 without port: 59


2825

## 6. Phân tích đặc trưng của Web Attack (TRAIN + VAL only)

So sánh phân phối một số feature giữa BENIGN, Web Brute Force, XSS và SQL Injection; không dùng test để đề xuất feature.

In [88]:
focus_labels = [x for x in sorted(sample.Label.unique()) if x=='BENIGN' or 'Web Attack' in x]
focus = sample.loc[~sample._split.eq('test') & sample.Label.isin(focus_labels), ['Label','_split']].copy()
focus['Label'] = focus['Label'].astype(str)
print('Focus support (train + val only):')
display(pd.crosstab(focus['Label'], focus['_split']))

probe_features = [f for f in ['Destination Port','Flow Duration','Total Fwd Packets','Total Backward Packets',
                              'Fwd Packet Length Mean','Bwd Packet Length Mean','Flow IAT Mean',
                              'Packet Length Mean','Init_Win_bytes_forward','Subflow Fwd Bytes']
                  if f in selected]
rows=[]
for label in focus_labels:
    m = sample.Label.eq(label).to_numpy() & ~sample._split.eq('test').to_numpy()
    for f in probe_features:
        arr = X_raw.loc[m,f]
        rows.append({'label':label,'feature':f,'n_nonmissing':int(arr.notna().sum()),
                     'median':float(arr.median()) if arr.notna().any() else None,
                     'q25':float(arr.quantile(.25)) if arr.notna().any() else None,
                     'q75':float(arr.quantile(.75)) if arr.notna().any() else None})
profile = pd.DataFrame(rows)
profile.to_csv(OUTPUT/'web_attack_feature_profiles_train_val.csv',index=False)
display(profile.head(30))

Focus support (train + val only):


_split,train,val
Label,,
BENIGN,4279,1368
Web Attack ï¿½ Brute Force,945,269
Web Attack ï¿½ Sql Injection,16,2
Web Attack ï¿½ XSS,388,130


,label,feature,n_nonmissing,median,q25,q75
0,BENIGN,Destination Port,5647,8.000000e+01,53.00,4.430000e+02
1,BENIGN,Flow Duration,5647,3.099300e+04,176.00,4.450945e+05
2,BENIGN,Total Fwd Packets,5647,2.000000e+00,2.00,4.000000e+00
3,BENIGN,Total Backward Packets,5647,2.000000e+00,1.00,2.000000e+00
4,BENIGN,Fwd Packet Length Mean,5647,3.700000e+01,6.00,5.100000e+01
5,BENIGN,Bwd Packet Length Mean,5647,8.000000e+01,4.90,1.550000e+02
6,BENIGN,Flow IAT Mean,5647,1.054333e+04,67.50,9.963529e+04
7,BENIGN,Packet Length Mean,5647,5.968966e+01,6.00,9.700000e+01
8,BENIGN,Init_Win_bytes_forward,5647,1.140000e+02,-1.00,8.192000e+03
9,Web Attack ï¿½ Brute Force,Destination Port,1214,8.000000e+01,80.00,8.000000e+01


## 7. Baseline nhanh chỉ trên validation: all-features vs no-port

Random Forest là **probe cho feature utility**, không thay thế Notebook 04. Fit trên train, đánh giá validation; không xem test. **Chỉ số lớp hiếm có support 0 trên validation không có ý nghĩa đo lường**. Các notebook 04 V2 và 05 V6 cần chạy lại trên artifact mới để so sánh công bằng.

In [89]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, f1_score, classification_report
from sklearn.preprocessing import LabelEncoder
import joblib

encoder = LabelEncoder().fit(sample.loc[train_mask,'Label'])
y = encoder.transform(sample['Label'])
val_mask = sample._split.eq('val').to_numpy()
val_rows=[]
probes={}
for variant, fs in [('with_port',features_with_port),('without_port',features_without_port)]:
    pipeline=Pipeline([('imputer',SimpleImputer(strategy='median')),
                       ('rf',RandomForestClassifier(n_estimators=160,min_samples_leaf=2,
                                      class_weight='balanced_subsample',n_jobs=-1,random_state=SEED))])
    pipeline.fit(X_raw.loc[train_mask,fs],y[train_mask])
    val_pred = pipeline.predict(X_raw.loc[val_mask,fs])
    val_rows.append({'feature_set':variant,'num_features':len(fs),'accuracy':accuracy_score(y[val_mask],val_pred),
                     'macro_f1_all_15':f1_score(y[val_mask],val_pred,average='macro',labels=np.arange(len(encoder.classes_)),zero_division=0)})
    rep=classification_report(y[val_mask],val_pred,labels=np.arange(len(encoder.classes_)),
            target_names=list(encoder.classes_),output_dict=True,zero_division=0)
    (OUTPUT/f'{variant}_validation_report.json').write_text(json.dumps(rep,indent=2),encoding='utf-8')
    importances = pd.DataFrame({'feature':fs,'importance':pipeline.named_steps['rf'].feature_importances_}).sort_values('importance',ascending=False)
    importances.to_csv(OUTPUT/f'{variant}_train_feature_importance.csv',index=False)
    probes[variant]=pipeline
    print(variant,'validation Macro-F1:',round(val_rows[-1]['macro_f1_all_15'],4))
    display(importances.head(15))
val_metrics=pd.DataFrame(val_rows)
val_metrics.to_csv(OUTPUT/'feature_variant_validation_metrics.csv',index=False)
display(val_metrics)
best_variant=val_metrics.sort_values(['macro_f1_all_15','accuracy'],ascending=False).iloc[0]['feature_set']
print('Best feature variant by validation Macro-F1:',best_variant)

with_port validation Macro-F1: 0.8729


,feature,importance
0,Destination Port,0.079435
49,Init_Win_bytes_backward,0.055067
16,Flow IAT Mean,0.032790
34,Bwd Packets/s,0.032205
51,min_seg_size_forward,0.031703
46,Average Packet Size,0.028753
37,Packet Length Mean,0.028364
48,Init_Win_bytes_forward,0.027732
4,Total Length of Fwd Packets,0.026496
15,Flow Packets/s,0.023128


without_port validation Macro-F1: 0.8665


,feature,importance
48,Init_Win_bytes_backward,0.069977
50,min_seg_size_forward,0.035844
35,Max Packet Length,0.031350
5,Fwd Packet Length Max,0.031054
3,Total Length of Fwd Packets,0.030318
47,Init_Win_bytes_forward,0.029730
33,Bwd Packets/s,0.027865
15,Flow IAT Mean,0.027821
0,Flow Duration,0.027628
16,Flow IAT Std,0.027409


,feature_set,num_features,accuracy,macro_f1_all_15
0,with_port,60,0.978759,0.872936
1,without_port,59,0.976215,0.866479


Best feature variant by validation Macro-F1: with_port


## 8. Xuất split và manifest dùng chung cho Notebook 04 V2 / 05 V6

Lưu **tất cả** feature đã lọc trên train, split, nhãn gốc và định danh nguồn. `test` vẫn tồn tại trong artifact để đánh giá cuối theo quy trình sau, **nhưng notebook 03B chưa huấn luyện/chấm điểm lên test**.

Bản Parquet mới KHÔNG tương thích trực tiếp với notebook 04/05 cũ nếu các notebook đó tự nhận mọi cột không phải `Label` là feature. Notebook 04 V2 / 05 V6 **phải** đọc `feature_sets.json`, loại `_source_*`, `_split`, `_duplicate_group`, `_binary_label`.

In [90]:
# Save both variant feature sets in one artifact; downstream must select by feature_sets.json.
export_columns = selected + ['Label','_binary_label','_split','_source_file','_source_row','_duplicate_group']
export = X_raw[selected].copy()
for col in ['Label','_binary_label','_split','_source_file','_source_row','_duplicate_group']:
    export[col]=sample[col].to_numpy()
output_path=OUTPUT/'cic_expanded_splits.parquet'
export.to_parquet(output_path,index=False)
assert len(pd.read_parquet(output_path,columns=['Label']))==len(sample)

manifest={
 'protocol':'CIC_FEATURE_EXPANDED_STRATIFIED_GROUP_5FOLD_PILOT_NO_IP_TIME',
 'warning':'New split: scores are NOT directly comparable to the old 19-feature split; random group-aware split does not eliminate event/host leakage.',
 'seed':SEED,'raw_columns':len(all_columns),'sample_rows':len(sample),
 'source_rows_by_label':dict(source_counts),
 'source_files':read_log,
 'split_counts':sample._split.value_counts().to_dict(),
 'train_class_support':support['train'].to_dict(),
 'validation_class_support':support['val'].to_dict(),
 'test_class_support':support['test'].to_dict(),
 'all_feature_candidates':raw_features,
 'feature_sets':{'with_port':features_with_port,'without_port':features_without_port},
 'validation_selected_feature_set':best_variant,
 'exact_feature_duplicate_groups':int(sample._duplicate_group.nunique()),
 'group_label_conflicts':int(conflicts.sum()),
 'data_file':output_path.name,
 'limitations':['class-balanced pilot does not reflect deployment prevalence',
                'test split belongs to the same captured dataset: no true independent holdout',
                'lack of IP/time prevents chronological and host-group splitting',
                'very rare classes may be missing from val/test folds'],
}
(OUTPUT/'experiment_manifest.json').write_text(json.dumps(manifest,ensure_ascii=False,indent=2),encoding='utf-8')
print('Saved:',output_path,'size(MB):',round(output_path.stat().st_size/1e6,1))
print('All outputs:',*[p.name for p in sorted(OUTPUT.iterdir())],sep='\n - ')

Saved: /kaggle/working/nids_outputs/03B_CIC_Feature_Engineering/cic_expanded_splits.parquet size(MB): 4.4
All outputs:
 - cic_expanded_splits.parquet
 - class_split_support.csv
 - experiment_manifest.json
 - feature_quality_audit.csv
 - feature_sets.json
 - feature_variant_validation_metrics.csv
 - web_attack_feature_profiles_train_val.csv
 - with_port_train_feature_importance.csv
 - with_port_validation_report.json
 - without_port_train_feature_importance.csv
 - without_port_validation_report.json
